# Phase 3: Fine-tuning Handcrafted Features + SVM

This notebook fine-tunes the four strongest phase 1 feature groups:

- Raw Pixels (Baseline)
- HOG + Color Histogram - gray
- HOG only - yuv
- HOG + Color Histogram - yuv

Scope follows `vault/finetuning.md` sections 1-3 only:

- SVM hyperparameter tuning, including optional SMOTE.
- Feature-specific tuning for raw pixels, HOG, HSV color histogram, and YUV channel weighting.
- Data-level development through padding-preserving resize and train-only augmentation.

Deep learning from section 4 is intentionally not implemented.

## Notes Before Running

The notebook reuses the existing `cropped_dataset/train`, `cropped_dataset/val`, and `cropped_dataset/test` folders. It does not create a new split and does not modify dataset files.

Default search mode is random search with a cap per feature group because the full grid is large. To run every candidate, set `SEARCH_MODE = 'grid'`.

Optional SMOTE requires `imbalanced-learn`. If it is not installed, SMOTE candidates are skipped automatically.

In [1]:
from pathlib import Path
from time import perf_counter
import itertools
import json
import math
import warnings

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from skimage.feature import hog
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    IMBLEARN_AVAILABLE = True
except ModuleNotFoundError:
    SMOTE = None
    ImbPipeline = None
    IMBLEARN_AVAILABLE = False

try:
    from tqdm.auto import tqdm
except ModuleNotFoundError:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else []

warnings.filterwarnings('ignore', category=UserWarning)
plt.style.use('ggplot')

RANDOM_STATE = 42
DATASET_DIR = Path('cropped_dataset')
PHASE1_RESULTS_DIR = Path('log/exphase_1_result')
PHASE1_METADATA_PATH = PHASE1_RESULTS_DIR / 'best_feature_svm_metadata.joblib'
PHASE1_VALIDATION_PATH = PHASE1_RESULTS_DIR / 'feature_svm_validation_results.csv'
RESULTS_DIR = Path('log/exphase_3_result')
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# Use 'random' for a manageable run, or 'grid' to run the full candidate set.
SEARCH_MODE = 'random'
MAX_CANDIDATES_PER_GROUP = 30
SAVE_AFTER_EACH_CANDIDATE = True
EVALUATE_TEST_FOR_EACH_GROUP = False

# Section 1: SVM tuning and optional data imbalance handling.
ENABLE_SMOTE_EXPERIMENTS = True
SMOTE_K_NEIGHBORS = 3

# Section 3: data-level development. These are in-memory only.
PREPROCESS_MODES = ['stretch', 'pad_square']
AUGMENTATION_MODES = ['none', 'minority_light']
AUGMENTATION_TARGET = 'median'
ENABLE_HORIZONTAL_FLIP = False  # Disabled by default because arrow signs may change meaning.

DEFAULT_TARGET_SIZE = (64, 64)  # cv2 uses (width, height)
PHASE1_HOG_PARAMS = {
    'orientations': 9,
    'pixels_per_cell': (8, 8),
    'cells_per_block': (2, 2),
    'block_norm': 'L2-Hys',
    'transform_sqrt': True,
    'feature_vector': True,
}
PHASE1_COLOR_HIST_BINS = (8, 8, 8)
PHASE1_SVM_PARAMS = {
    'kernel': 'rbf',
    'C': 10.0,
    'gamma': 'scale',
    'class_weight': 'balanced',
    'cache_size': 1024,
}

phase1_metadata = {}
if PHASE1_METADATA_PATH.exists():
    phase1_metadata = joblib.load(PHASE1_METADATA_PATH)
    print(f'Loaded phase 1 metadata from {PHASE1_METADATA_PATH}')
else:
    print(f'Phase 1 metadata not found at {PHASE1_METADATA_PATH}; using notebook defaults.')

TARGET_SIZE = tuple(phase1_metadata.get('target_size', DEFAULT_TARGET_SIZE))
BASE_HOG_PARAMS = dict(phase1_metadata.get('hog_params', PHASE1_HOG_PARAMS))
BASE_COLOR_HIST_BINS = tuple(phase1_metadata.get('color_hist_bins', PHASE1_COLOR_HIST_BINS))
BASE_SVM_PARAMS = dict(phase1_metadata.get('svm_params', PHASE1_SVM_PARAMS))

np.random.seed(RANDOM_STATE)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Phase 3 config ready')
print('Search mode:', SEARCH_MODE)
print('Max candidates per group:', MAX_CANDIDATES_PER_GROUP)
print('SMOTE available:', IMBLEARN_AVAILABLE)
print('Results dir:', RESULTS_DIR)

Loaded phase 1 metadata from log/exphase_1_result/best_feature_svm_metadata.joblib
Phase 3 config ready
Search mode: random
Max candidates per group: 30
SMOTE available: False
Results dir: log/exphase_3_result


/home/dacekey/AIL303_SUM26/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def read_image_bgr(path: Path):
    data = np.fromfile(str(path), dtype=np.uint8)
    return cv2.imdecode(data, cv2.IMREAD_COLOR)


def get_class_names(dataset_dir: Path) -> list[str]:
    train_dir = dataset_dir / 'train'
    if not train_dir.exists():
        raise FileNotFoundError(f'Missing training directory: {train_dir}')
    return sorted(path.name for path in train_dir.iterdir() if path.is_dir())


def collect_split_paths(dataset_dir: Path, split: str, class_names: list[str]) -> tuple[list[Path], np.ndarray]:
    split_dir = dataset_dir / split
    if not split_dir.exists():
        raise FileNotFoundError(f'Missing split directory: {split_dir}')

    paths = []
    labels = []
    for label, class_name in enumerate(class_names):
        class_dir = split_dir / class_name
        if not class_dir.exists():
            print(f'Warning: missing class folder in {split}: {class_name}')
            continue
        class_paths = [p for p in sorted(class_dir.iterdir()) if p.suffix.lower() in IMAGE_EXTENSIONS]
        paths.extend(class_paths)
        labels.extend([label] * len(class_paths))
    return paths, np.asarray(labels, dtype=np.int64)


def load_split(dataset_dir: Path, split: str, class_names: list[str]) -> tuple[list[np.ndarray], np.ndarray, list[Path]]:
    paths, labels = collect_split_paths(dataset_dir, split, class_names)
    images = []
    kept_labels = []
    kept_paths = []
    bad_paths = []

    for path, label in tqdm(list(zip(paths, labels)), desc=f'Loading {split}'):
        image = read_image_bgr(path)
        if image is None:
            bad_paths.append(path)
            continue
        images.append(image)
        kept_labels.append(label)
        kept_paths.append(path)

    if bad_paths:
        print(f'Skipped {len(bad_paths)} unreadable images from {split}')
    if not images:
        raise ValueError(f'No images loaded for split: {split}')

    return images, np.asarray(kept_labels, dtype=np.int64), kept_paths


def summarize_labels(y: np.ndarray, class_names: list[str], split: str) -> pd.DataFrame:
    counts = pd.Series(y).value_counts().sort_index()
    return pd.DataFrame({
        'split': split,
        'class_id': counts.index,
        'class_name': [class_names[i] for i in counts.index],
        'count': counts.values,
    })


folder_class_names = get_class_names(DATASET_DIR)
phase1_class_names = phase1_metadata.get('class_names')

if phase1_class_names is not None:
    class_names = list(phase1_class_names)
    if set(class_names) != set(folder_class_names):
        missing = sorted(set(class_names) - set(folder_class_names))
        extra = sorted(set(folder_class_names) - set(class_names))
        raise ValueError(f'Class mismatch. Missing={missing[:5]}, extra={extra[:5]}')
    print('Using phase 1 label order from metadata.')
else:
    class_names = folder_class_names
    print('Using sorted train folder names as label order.')

data = {}
for split in ['train', 'val', 'test']:
    images, labels, paths = load_split(DATASET_DIR, split, class_names)
    data[split] = {'images': images, 'y': labels, 'paths': paths}
    print(f'{split}: {len(images)} images')

label_summary = pd.concat(
    [summarize_labels(data[split]['y'], class_names, split) for split in ['train', 'val', 'test']],
    ignore_index=True,
)
display(label_summary.groupby('split')['count'].agg(['sum', 'min', 'median', 'max']))

Using phase 1 label order from metadata.


Loading train: 100%|███████████████| 6605/6605 [00:01<00:00, 5836.77it/s]


train: 6605 images


Loading val: 100%|███████████████████| 824/824 [00:00<00:00, 5173.56it/s]


val: 824 images


Loading test: 100%|██████████████████| 851/851 [00:00<00:00, 5105.49it/s]

test: 851 images


,sum,min,median,max
split,,,,
test,851,3,11.0,108
train,6605,16,84.0,856
val,824,2,10.0,107


In [3]:
def resize_stretch_bgr(image_bgr: np.ndarray, size: int) -> np.ndarray:
    height, width = image_bgr.shape[:2]
    interpolation = cv2.INTER_AREA if height > size or width > size else cv2.INTER_CUBIC
    return cv2.resize(image_bgr, (size, size), interpolation=interpolation)


def resize_pad_square_bgr(image_bgr: np.ndarray, size: int) -> np.ndarray:
    height, width = image_bgr.shape[:2]
    side = max(height, width)
    canvas = np.zeros((side, side, 3), dtype=image_bgr.dtype)
    y0 = (side - height) // 2
    x0 = (side - width) // 2
    canvas[y0:y0 + height, x0:x0 + width] = image_bgr
    interpolation = cv2.INTER_AREA if side > size else cv2.INTER_CUBIC
    return cv2.resize(canvas, (size, size), interpolation=interpolation)


def preprocess_image_bgr(image_bgr: np.ndarray, size: int, mode: str) -> np.ndarray:
    if mode == 'stretch':
        return resize_stretch_bgr(image_bgr, size)
    if mode == 'pad_square':
        return resize_pad_square_bgr(image_bgr, size)
    raise ValueError(f'Unknown preprocess mode: {mode}')


def augment_image_bgr(image_bgr: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    image = image_bgr.copy()

    if ENABLE_HORIZONTAL_FLIP and rng.random() < 0.25:
        image = cv2.flip(image, 1)

    height, width = image.shape[:2]
    angle = float(rng.uniform(-8.0, 8.0))
    scale = float(rng.uniform(0.95, 1.05))
    matrix = cv2.getRotationMatrix2D((width / 2, height / 2), angle, scale)
    image = cv2.warpAffine(
        image,
        matrix,
        (width, height),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0),
    )

    alpha = float(rng.uniform(0.85, 1.15))
    beta = float(rng.uniform(-12.0, 12.0))
    image = cv2.convertScaleAbs(image, alpha=alpha, beta=beta)

    if rng.random() < 0.35:
        noise = rng.normal(0, 5, image.shape).astype(np.float32)
        image = np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    return image


def augment_minority_train_images(
    images: list[np.ndarray],
    y: np.ndarray,
    strategy: str,
    seed: int = RANDOM_STATE,
) -> tuple[list[np.ndarray], np.ndarray]:
    if strategy == 'none':
        return images, y
    if strategy != 'minority_light':
        raise ValueError(f'Unknown augmentation strategy: {strategy}')

    rng = np.random.default_rng(seed)
    counts = pd.Series(y).value_counts().sort_index()
    if AUGMENTATION_TARGET == 'median':
        target_count = int(np.median(counts.values))
    elif AUGMENTATION_TARGET == 'mean':
        target_count = int(np.mean(counts.values))
    else:
        target_count = int(AUGMENTATION_TARGET)

    augmented_images = list(images)
    augmented_labels = list(y)

    for class_id, count in counts.items():
        if count >= target_count:
            continue
        class_indices = np.where(y == class_id)[0]
        needed = target_count - int(count)
        for idx in range(needed):
            source_image = images[int(class_indices[idx % len(class_indices)])]
            augmented_images.append(augment_image_bgr(source_image, rng))
            augmented_labels.append(int(class_id))

    order = rng.permutation(len(augmented_labels))
    shuffled_images = [augmented_images[int(i)] for i in order]
    shuffled_labels = np.asarray([augmented_labels[int(i)] for i in order], dtype=np.int64)
    return shuffled_images, shuffled_labels


IMAGE_CACHE = {}


def get_preprocessed_images(split: str, size: int, preprocess_mode: str, augmentation_mode: str):
    if split != 'train' and augmentation_mode != 'none':
        augmentation_mode = 'none'

    cache_key = (split, size, preprocess_mode, augmentation_mode)
    if cache_key in IMAGE_CACHE:
        return IMAGE_CACHE[cache_key]

    images = data[split]['images']
    labels = data[split]['y']
    if split == 'train':
        images, labels = augment_minority_train_images(images, labels, augmentation_mode)

    processed = [
        preprocess_image_bgr(image, size=size, mode=preprocess_mode)
        for image in tqdm(images, desc=f'{split}/{preprocess_mode}/{size}/{augmentation_mode}')
    ]
    processed = np.stack(processed).astype(np.uint8)
    labels = np.asarray(labels, dtype=np.int64)

    IMAGE_CACHE[cache_key] = (processed, labels)
    print(f'{cache_key}: images={processed.shape}, labels={labels.shape}')
    return IMAGE_CACHE[cache_key]

In [4]:
def as_gray_float(image_bgr: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    return gray.astype(np.float32) / 255.0


def as_yuv_float_channels(image_bgr: np.ndarray) -> list[np.ndarray]:
    yuv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2YUV)
    return [yuv[:, :, channel].astype(np.float32) / 255.0 for channel in range(3)]


def hog_channel(channel_float: np.ndarray, hog_params: dict) -> np.ndarray:
    return hog(channel_float, **hog_params).astype(np.float32)


def extract_raw_gray_block(images_bgr: np.ndarray) -> tuple[np.ndarray, list[tuple[str, int, int]], dict[str, float]]:
    features = [as_gray_float(image).ravel() for image in tqdm(images_bgr, desc='Raw gray')]
    matrix = np.vstack(features).astype(np.float32)
    blocks = [('raw_pixels', 0, matrix.shape[1])]
    weights = {'raw_pixels': 1.0}
    return matrix, blocks, weights


def extract_hog_gray_block(
    images_bgr: np.ndarray,
    hog_params: dict,
) -> tuple[np.ndarray, list[tuple[str, int, int]], dict[str, float]]:
    features = [hog_channel(as_gray_float(image), hog_params) for image in tqdm(images_bgr, desc='HOG gray')]
    matrix = np.vstack(features).astype(np.float32)
    blocks = [('hog_gray', 0, matrix.shape[1])]
    weights = {'hog_gray': 1.0}
    return matrix, blocks, weights


def extract_hog_yuv_blocks(
    images_bgr: np.ndarray,
    hog_params: dict,
    yuv_weights: tuple[float, float, float],
) -> tuple[np.ndarray, list[tuple[str, int, int]], dict[str, float]]:
    y_features = []
    u_features = []
    v_features = []
    for image in tqdm(images_bgr, desc='HOG YUV'):
        y_channel, u_channel, v_channel = as_yuv_float_channels(image)
        y_features.append(hog_channel(y_channel, hog_params))
        u_features.append(hog_channel(u_channel, hog_params))
        v_features.append(hog_channel(v_channel, hog_params))

    y_matrix = np.vstack(y_features).astype(np.float32)
    u_matrix = np.vstack(u_features).astype(np.float32)
    v_matrix = np.vstack(v_features).astype(np.float32)
    matrix = np.hstack([y_matrix, u_matrix, v_matrix]).astype(np.float32)

    y_end = y_matrix.shape[1]
    u_end = y_end + u_matrix.shape[1]
    v_end = u_end + v_matrix.shape[1]
    blocks = [('hog_y', 0, y_end), ('hog_u', y_end, u_end), ('hog_v', u_end, v_end)]
    weights = {'hog_y': yuv_weights[0], 'hog_u': yuv_weights[1], 'hog_v': yuv_weights[2]}
    return matrix, blocks, weights


def extract_hsv_histogram_block(
    images_bgr: np.ndarray,
    bins: tuple[int, int, int],
) -> tuple[np.ndarray, list[tuple[str, int, int]], dict[str, float]]:
    features = []
    hist_ranges = [0, 180, 0, 256, 0, 256]
    for image in tqdm(images_bgr, desc=f'HSV hist {bins}'):
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
        hist = cv2.calcHist([hsv], [0, 1, 2], None, bins, hist_ranges).astype(np.float32)
        hist = hist.ravel()
        hist /= hist.sum() + 1e-8
        features.append(hist)
    matrix = np.vstack(features).astype(np.float32)
    blocks = [('hist_hsv', 0, matrix.shape[1])]
    weights = {'hist_hsv': 1.0}
    return matrix, blocks, weights


def offset_blocks(blocks: list[tuple[str, int, int]], offset: int) -> list[tuple[str, int, int]]:
    return [(name, start + offset, end + offset) for name, start, end in blocks]


def combine_feature_outputs(outputs):
    matrices = []
    blocks = []
    weights = {}
    offset = 0
    for matrix, matrix_blocks, matrix_weights in outputs:
        matrices.append(matrix)
        blocks.extend(offset_blocks(matrix_blocks, offset))
        weights.update(matrix_weights)
        offset += matrix.shape[1]
    return np.hstack(matrices).astype(np.float32), blocks, weights


def extract_feature_blocks(images_bgr: np.ndarray, feature_config: dict):
    feature_type = feature_config['feature_type']
    if feature_type == 'raw_gray':
        return extract_raw_gray_block(images_bgr)

    hog_params = dict(feature_config.get('hog_params', BASE_HOG_PARAMS))
    hist_bins = tuple(feature_config.get('hist_bins', BASE_COLOR_HIST_BINS))
    yuv_weights = tuple(feature_config.get('yuv_weights', (1.0, 1.0, 1.0)))

    if feature_type == 'hog_gray_hist':
        return combine_feature_outputs([
            extract_hog_gray_block(images_bgr, hog_params),
            extract_hsv_histogram_block(images_bgr, hist_bins),
        ])
    if feature_type == 'hog_yuv':
        return extract_hog_yuv_blocks(images_bgr, hog_params, yuv_weights)
    if feature_type == 'hog_yuv_hist':
        return combine_feature_outputs([
            extract_hog_yuv_blocks(images_bgr, hog_params, yuv_weights),
            extract_hsv_histogram_block(images_bgr, hist_bins),
        ])

    raise ValueError(f'Unknown feature type: {feature_type}')


FEATURE_CACHE = {}
FEATURE_TIMINGS = {}


def stable_json(value) -> str:
    return json.dumps(value, sort_keys=True, default=str)


def get_candidate_features(spec: dict, split: str):
    data_config = spec['data_config']
    feature_config = spec['feature_config']
    augmentation_mode = data_config['augmentation_mode'] if split == 'train' else 'none'
    cache_key = (
        split,
        data_config['image_size'],
        data_config['preprocess_mode'],
        augmentation_mode,
        stable_json(feature_config),
    )
    if cache_key in FEATURE_CACHE:
        return FEATURE_CACHE[cache_key]

    images, labels = get_preprocessed_images(
        split=split,
        size=data_config['image_size'],
        preprocess_mode=data_config['preprocess_mode'],
        augmentation_mode=augmentation_mode,
    )
    started = perf_counter()
    matrix, blocks, block_weights = extract_feature_blocks(images, feature_config)
    elapsed = perf_counter() - started

    FEATURE_CACHE[cache_key] = (matrix, labels, blocks, block_weights)
    FEATURE_TIMINGS[cache_key] = elapsed
    print(f'{split}/{spec["case"]}: X={matrix.shape}, feature_time_sec={elapsed:.3f}')
    return FEATURE_CACHE[cache_key]

In [5]:
class FeatureBlockScaler(BaseEstimator, TransformerMixin):
    def __init__(self, blocks, scaler_type='standard', block_weights=None):
        self.blocks = blocks
        self.scaler_type = scaler_type
        self.block_weights = block_weights

    def _make_scaler(self):
        if self.scaler_type == 'standard':
            return StandardScaler()
        if self.scaler_type == 'minmax':
            return MinMaxScaler()
        raise ValueError(f'Unknown scaler type: {self.scaler_type}')

    def fit(self, X, y=None):
        self.scalers_ = {}
        for name, start, end in self.blocks:
            scaler = self._make_scaler()
            scaler.fit(X[:, start:end])
            self.scalers_[name] = scaler
        return self

    def transform(self, X):
        weights = self.block_weights or {}
        matrices = []
        for name, start, end in self.blocks:
            block = self.scalers_[name].transform(X[:, start:end])
            block = block * float(weights.get(name, 1.0))
            matrices.append(block)
        return np.hstack(matrices).astype(np.float32)


def build_pipeline(spec: dict, blocks, block_weights):
    model_config = spec['model_config']
    steps = [
        ('block_scaler', FeatureBlockScaler(
            blocks=blocks,
            scaler_type=model_config['scaler_type'],
            block_weights=block_weights,
        )),
    ]

    pca_n_components = model_config.get('pca_n_components')
    if pca_n_components is not None:
        steps.append(('pca', PCA(
            n_components=pca_n_components,
            svd_solver='full',
            random_state=RANDOM_STATE,
        )))

    use_smote = bool(model_config.get('use_smote', False))
    if use_smote:
        if not IMBLEARN_AVAILABLE:
            raise RuntimeError('SMOTE requested but imbalanced-learn is not installed.')
        steps.append(('smote', SMOTE(
            random_state=RANDOM_STATE,
            k_neighbors=SMOTE_K_NEIGHBORS,
        )))

    steps.append(('svm', SVC(**model_config['svm_params'])))
    pipeline_class = ImbPipeline if use_smote else Pipeline
    return pipeline_class(steps)


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }


def slugify(text: str) -> str:
    chars = []
    for char in text.lower():
        chars.append(char if char.isalnum() else '_')
    return '_'.join(''.join(chars).split('_')).strip('_')


def top_confusions(y_true: np.ndarray, y_pred: np.ndarray, top_n: int = 20) -> pd.DataFrame:
    columns = ['true_class', 'predicted_class', 'count']
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
    rows = []
    for true_id in range(cm.shape[0]):
        for pred_id in range(cm.shape[1]):
            if true_id == pred_id or cm[true_id, pred_id] == 0:
                continue
            rows.append({
                'true_class': class_names[true_id],
                'predicted_class': class_names[pred_id],
                'count': int(cm[true_id, pred_id]),
            })
    if not rows:
        return pd.DataFrame(columns=columns)
    return pd.DataFrame(rows, columns=columns).sort_values('count', ascending=False).head(top_n)


def save_classification_outputs(prefix: str, y_true: np.ndarray, y_pred: np.ndarray):
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        zero_division=0,
        output_dict=True,
    )
    report_text = classification_report(
        y_true,
        y_pred,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        zero_division=0,
    )
    report_csv_path = RESULTS_DIR / f'{prefix}_classification_report.csv'
    report_txt_path = RESULTS_DIR / f'{prefix}_classification_report.txt'
    pd.DataFrame(report_dict).transpose().to_csv(report_csv_path)
    report_txt_path.write_text(report_text, encoding='utf-8')

    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
    cm_path = RESULTS_DIR / f'{prefix}_confusion_matrix.csv'
    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(cm_path, index_label='true_class')

    top_path = RESULTS_DIR / f'{prefix}_top_confusions.csv'
    top_confusions(y_true, y_pred).to_csv(top_path, index=False)

    return {
        'classification_report_csv': report_csv_path,
        'classification_report_txt': report_txt_path,
        'confusion_matrix_csv': cm_path,
        'top_confusions_csv': top_path,
    }

In [6]:
RAW_IMAGE_SIZES = [32, 48, 64, 128]
RAW_SCALERS = ['standard', 'minmax']
RAW_PCA_OPTIONS = [None, 0.95]

HOG_ORIENTATIONS = [8, 9, 12]
HOG_PIXELS_PER_CELL = [(4, 4), (8, 8)]
HOG_CELLS_PER_BLOCK = [(1, 1), (2, 2)]
COLOR_HIST_BINS_OPTIONS = [(8, 8, 8), (16, 16, 16), (32, 32, 32)]
YUV_WEIGHT_OPTIONS = [
    (1.0, 1.0, 1.0),
    (1.5, 1.0, 1.0),
    (2.0, 0.75, 0.75),
]

SVM_SEARCH_SPACE = [
    {
        'kernel': ['linear'],
        'C': [0.1, 1.0, 10.0],
        'class_weight': ['balanced', None],
        'cache_size': [1024],
    },
    {
        'kernel': ['rbf'],
        'C': [1.0, 3.0, 10.0, 30.0],
        'gamma': ['scale', 'auto', 0.01, 0.001],
        'class_weight': ['balanced', None],
        'cache_size': [1024],
    },
    {
        'kernel': ['poly'],
        'C': [1.0, 10.0],
        'degree': [2, 3],
        'gamma': ['scale'],
        'coef0': [0.0, 1.0],
        'class_weight': ['balanced'],
        'cache_size': [1024],
    },
]


def unique_dicts(dicts: list[dict]) -> list[dict]:
    seen = set()
    unique = []
    for item in dicts:
        key = stable_json(item)
        if key in seen:
            continue
        seen.add(key)
        unique.append(item)
    return unique


def expand_svm_candidates() -> list[dict]:
    candidates = [dict(BASE_SVM_PARAMS)]
    for grid in SVM_SEARCH_SPACE:
        candidates.extend(dict(params) for params in ParameterGrid(grid))
    return unique_dicts(candidates)


SVM_CANDIDATES = expand_svm_candidates()
SMOTE_OPTIONS = [False, True] if ENABLE_SMOTE_EXPERIMENTS and IMBLEARN_AVAILABLE else [False]
if ENABLE_SMOTE_EXPERIMENTS and not IMBLEARN_AVAILABLE:
    print('imbalanced-learn is not installed; SMOTE candidates will be skipped.')


def make_hog_params(orientations: int, pixels_per_cell: tuple[int, int], cells_per_block: tuple[int, int]) -> dict:
    params = dict(BASE_HOG_PARAMS)
    params.update({
        'orientations': orientations,
        'pixels_per_cell': pixels_per_cell,
        'cells_per_block': cells_per_block,
        'block_norm': 'L2-Hys',
        'transform_sqrt': True,
        'feature_vector': True,
    })
    return params


def model_configs_for_group(group_name: str) -> list[dict]:
    configs = []
    scaler_options = RAW_SCALERS if group_name == 'Raw Pixels (Baseline)' else ['standard']
    pca_options = RAW_PCA_OPTIONS if group_name == 'Raw Pixels (Baseline)' else [None]

    for scaler_type, pca_n_components, use_smote, svm_params in itertools.product(
        scaler_options,
        pca_options,
        SMOTE_OPTIONS,
        SVM_CANDIDATES,
    ):
        configs.append({
            'scaler_type': scaler_type,
            'pca_n_components': pca_n_components,
            'use_smote': use_smote,
            'svm_params': dict(svm_params),
        })
    return configs


def common_data_configs(image_sizes: list[int]) -> list[dict]:
    configs = []
    for image_size, preprocess_mode, augmentation_mode in itertools.product(
        image_sizes,
        PREPROCESS_MODES,
        AUGMENTATION_MODES,
    ):
        configs.append({
            'image_size': image_size,
            'preprocess_mode': preprocess_mode,
            'augmentation_mode': augmentation_mode,
        })
    return configs


def baseline_spec(case: str) -> dict:
    if case == 'Raw Pixels (Baseline)':
        feature_config = {'feature_type': 'raw_gray'}
    elif case == 'HOG + Color Histogram - gray':
        feature_config = {
            'feature_type': 'hog_gray_hist',
            'hog_params': dict(BASE_HOG_PARAMS),
            'hist_bins': BASE_COLOR_HIST_BINS,
        }
    elif case == 'HOG only - yuv':
        feature_config = {
            'feature_type': 'hog_yuv',
            'hog_params': dict(BASE_HOG_PARAMS),
            'yuv_weights': (1.0, 1.0, 1.0),
        }
    elif case == 'HOG + Color Histogram - yuv':
        feature_config = {
            'feature_type': 'hog_yuv_hist',
            'hog_params': dict(BASE_HOG_PARAMS),
            'hist_bins': BASE_COLOR_HIST_BINS,
            'yuv_weights': (1.0, 1.0, 1.0),
        }
    else:
        raise ValueError(case)

    return {
        'case': case,
        'feature_config': feature_config,
        'data_config': {
            'image_size': 64,
            'preprocess_mode': 'stretch',
            'augmentation_mode': 'none',
        },
        'model_config': {
            'scaler_type': 'standard',
            'pca_n_components': None,
            'use_smote': False,
            'svm_params': dict(BASE_SVM_PARAMS),
        },
        'baseline_like': True,
    }


def build_specs_for_group(case: str) -> list[dict]:
    specs = [baseline_spec(case)]

    if case == 'Raw Pixels (Baseline)':
        for data_config, model_config in itertools.product(
            common_data_configs(RAW_IMAGE_SIZES),
            model_configs_for_group(case),
        ):
            specs.append({
                'case': case,
                'feature_config': {'feature_type': 'raw_gray'},
                'data_config': data_config,
                'model_config': model_config,
                'baseline_like': False,
            })
        return specs

    hog_param_grid = [
        make_hog_params(orientations, pixels_per_cell, cells_per_block)
        for orientations, pixels_per_cell, cells_per_block in itertools.product(
            HOG_ORIENTATIONS,
            HOG_PIXELS_PER_CELL,
            HOG_CELLS_PER_BLOCK,
        )
    ]
    model_configs = model_configs_for_group(case)
    data_configs = common_data_configs([64])

    if case == 'HOG + Color Histogram - gray':
        for hog_params, hist_bins, data_config, model_config in itertools.product(
            hog_param_grid,
            COLOR_HIST_BINS_OPTIONS,
            data_configs,
            model_configs,
        ):
            specs.append({
                'case': case,
                'feature_config': {
                    'feature_type': 'hog_gray_hist',
                    'hog_params': hog_params,
                    'hist_bins': hist_bins,
                },
                'data_config': data_config,
                'model_config': model_config,
                'baseline_like': False,
            })
    elif case == 'HOG only - yuv':
        for hog_params, yuv_weights, data_config, model_config in itertools.product(
            hog_param_grid,
            YUV_WEIGHT_OPTIONS,
            data_configs,
            model_configs,
        ):
            specs.append({
                'case': case,
                'feature_config': {
                    'feature_type': 'hog_yuv',
                    'hog_params': hog_params,
                    'yuv_weights': yuv_weights,
                },
                'data_config': data_config,
                'model_config': model_config,
                'baseline_like': False,
            })
    elif case == 'HOG + Color Histogram - yuv':
        for hog_params, hist_bins, yuv_weights, data_config, model_config in itertools.product(
            hog_param_grid,
            COLOR_HIST_BINS_OPTIONS,
            YUV_WEIGHT_OPTIONS,
            data_configs,
            model_configs,
        ):
            specs.append({
                'case': case,
                'feature_config': {
                    'feature_type': 'hog_yuv_hist',
                    'hog_params': hog_params,
                    'hist_bins': hist_bins,
                    'yuv_weights': yuv_weights,
                },
                'data_config': data_config,
                'model_config': model_config,
                'baseline_like': False,
            })
    else:
        raise ValueError(case)

    return specs


def spec_signature(spec: dict) -> str:
    comparable = {key: value for key, value in spec.items() if key != 'candidate_id'}
    return stable_json(comparable)


def unique_specs(specs: list[dict]) -> list[dict]:
    seen = set()
    unique = []
    for spec in specs:
        key = spec_signature(spec)
        if key in seen:
            continue
        seen.add(key)
        unique.append(spec)
    return unique


def select_specs(specs: list[dict], max_candidates: int, seed: int) -> list[dict]:
    specs = unique_specs(specs)
    baseline = [spec for spec in specs if spec.get('baseline_like')]
    others = [spec for spec in specs if not spec.get('baseline_like')]

    if SEARCH_MODE == 'grid' or len(specs) <= max_candidates:
        selected = baseline + others
    elif SEARCH_MODE == 'random':
        rng = np.random.default_rng(seed)
        take = max(0, max_candidates - len(baseline))
        if take >= len(others):
            sampled = others
        else:
            indices = rng.choice(len(others), size=take, replace=False)
            sampled = [others[int(i)] for i in indices]
        selected = baseline + sampled
    else:
        raise ValueError(f'Unknown SEARCH_MODE: {SEARCH_MODE}')

    return unique_specs(selected)


SELECTED_CASES = [
    'Raw Pixels (Baseline)',
    'HOG + Color Histogram - gray',
    'HOG only - yuv',
    'HOG + Color Histogram - yuv',
]

search_plan = []
for group_idx, case in enumerate(SELECTED_CASES):
    group_specs = build_specs_for_group(case)
    selected = select_specs(
        group_specs,
        max_candidates=MAX_CANDIDATES_PER_GROUP,
        seed=RANDOM_STATE + group_idx,
    )
    print(f'{case}: selected {len(selected)} / {len(unique_specs(group_specs))} candidates')
    search_plan.extend(selected)

for idx, spec in enumerate(search_plan):
    spec['candidate_id'] = f'{slugify(spec["case"])}_{idx:04d}'


def flatten_spec(spec: dict) -> dict:
    feature_config = spec['feature_config']
    data_config = spec['data_config']
    model_config = spec['model_config']
    hog_params = feature_config.get('hog_params', {})
    svm_params = model_config['svm_params']
    return {
        'candidate_id': spec['candidate_id'],
        'case': spec['case'],
        'feature_type': feature_config['feature_type'],
        'image_size': data_config['image_size'],
        'preprocess_mode': data_config['preprocess_mode'],
        'augmentation_mode': data_config['augmentation_mode'],
        'hog_orientations': hog_params.get('orientations'),
        'hog_pixels_per_cell': hog_params.get('pixels_per_cell'),
        'hog_cells_per_block': hog_params.get('cells_per_block'),
        'hist_bins': feature_config.get('hist_bins'),
        'yuv_weights': feature_config.get('yuv_weights'),
        'scaler_type': model_config['scaler_type'],
        'pca_n_components': model_config.get('pca_n_components'),
        'use_smote': model_config.get('use_smote', False),
        'svm_kernel': svm_params.get('kernel'),
        'svm_C': svm_params.get('C'),
        'svm_gamma': svm_params.get('gamma'),
        'svm_degree': svm_params.get('degree'),
        'svm_coef0': svm_params.get('coef0'),
        'svm_class_weight': svm_params.get('class_weight'),
        'baseline_like': spec.get('baseline_like', False),
    }


search_plan_df = pd.DataFrame([flatten_spec(spec) for spec in search_plan])
search_plan_path = RESULTS_DIR / 'phase3_search_plan.csv'
search_plan_df.to_csv(search_plan_path, index=False)
print(f'Saved search plan to {search_plan_path}')
display(search_plan_df.head(20))

imbalanced-learn is not installed; SMOTE candidates will be skipped.
Raw Pixels (Baseline): selected 30 / 2945 candidates
HOG + Color Histogram - gray: selected 30 / 6625 candidates
HOG only - yuv: selected 30 / 6625 candidates
HOG + Color Histogram - yuv: selected 30 / 19873 candidates
Saved search plan to log/exphase_3_result/phase3_search_plan.csv


,candidate_id,case,feature_type,image_size,preprocess_mode,augmentation_mode,hog_orientations,hog_pixels_per_cell,hog_cells_per_block,hist_bins,...,scaler_type,pca_n_components,use_smote,svm_kernel,svm_C,svm_gamma,svm_degree,svm_coef0,svm_class_weight,baseline_like
0,raw_pixels__baseline_0000,Raw Pixels (Baseline),raw_gray,64,stretch,none,NaN,None,None,None,...,standard,NaN,False,rbf,10.0,scale,NaN,NaN,balanced,True
1,raw_pixels__baseline_0001,Raw Pixels (Baseline),raw_gray,48,pad_square,none,NaN,None,None,None,...,minmax,0.95,False,rbf,3.0,0.001,NaN,NaN,NaN,False
2,raw_pixels__baseline_0002,Raw Pixels (Baseline),raw_gray,48,pad_square,none,NaN,None,None,None,...,minmax,0.95,False,poly,1.0,scale,3.0,0.0,balanced,False
3,raw_pixels__baseline_0003,Raw Pixels (Baseline),raw_gray,128,stretch,none,NaN,None,None,None,...,standard,0.95,False,poly,10.0,scale,3.0,0.0,balanced,False
4,raw_pixels__baseline_0004,Raw Pixels (Baseline),raw_gray,32,pad_square,minority_light,NaN,None,None,None,...,standard,NaN,False,rbf,30.0,0.001,NaN,NaN,NaN,False
5,raw_pixels__baseline_0005,Raw Pixels (Baseline),raw_gray,128,stretch,none,NaN,None,None,None,...,minmax,NaN,False,linear,1.0,None,NaN,NaN,balanced,False
6,raw_pixels__baseline_0006,Raw Pixels (Baseline),raw_gray,48,pad_square,minority_light,NaN,None,None,None,...,minmax,0.95,False,poly,10.0,scale,3.0,0.0,balanced,False
7,raw_pixels__baseline_0007,Raw Pixels (Baseline),raw_gray,32,pad_square,none,NaN,None,None,None,...,minmax,0.95,False,rbf,30.0,scale,NaN,NaN,balanced,False
8,raw_pixels__baseline_0008,Raw Pixels (Baseline),raw_gray,64,stretch,none,NaN,None,None,None,...,standard,0.95,False,rbf,3.0,0.001,NaN,NaN,NaN,False
9,raw_pixels__baseline_0009,Raw Pixels (Baseline),raw_gray,64,pad_square,minority_light,NaN,None,None,None,...,standard,NaN,False,rbf,1.0,0.001,NaN,NaN,NaN,False


In [ ]:
def run_candidate(spec: dict) -> tuple[dict, Pipeline, np.ndarray]:
    train_feature_started = perf_counter()
    X_train, y_train, blocks, block_weights = get_candidate_features(spec, 'train')
    train_feature_time = perf_counter() - train_feature_started

    val_feature_started = perf_counter()
    X_val, y_val, _, _ = get_candidate_features(spec, 'val')
    val_feature_time = perf_counter() - val_feature_started

    model = build_pipeline(spec, blocks=blocks, block_weights=block_weights)
    fit_started = perf_counter()
    model.fit(X_train, y_train)
    fit_time = perf_counter() - fit_started

    predict_started = perf_counter()
    val_pred = model.predict(X_val)
    predict_time = perf_counter() - predict_started

    metrics = compute_metrics(y_val, val_pred)
    result = {
        **flatten_spec(spec),
        'n_features': X_train.shape[1],
        'train_samples': int(len(y_train)),
        'val_samples': int(len(y_val)),
        'feature_extraction_train_sec': train_feature_time,
        'feature_extraction_val_sec': val_feature_time,
        'fit_time_sec': fit_time,
        'train_time_sec': fit_time,
        'val_predict_sec': predict_time,
        'inference_time_ms_per_image': ((val_feature_time + predict_time) / len(y_val)) * 1000.0,
        **{f'val_{key}': value for key, value in metrics.items()},
    }
    return result, model, val_pred


validation_results = []
best_models = {}
best_val_predictions = {}

partial_path = RESULTS_DIR / 'phase3_validation_results_partial.csv'

for spec in tqdm(search_plan, desc='Phase 3 candidates'):
    print(f"\nRunning {spec['candidate_id']}: {spec['case']}")
    try:
        result, model, val_pred = run_candidate(spec)
    except Exception as exc:
        result = {
            **flatten_spec(spec),
            'error': repr(exc),
        }
        validation_results.append(result)
        print(f"Failed {spec['candidate_id']}: {exc}")
        if SAVE_AFTER_EACH_CANDIDATE:
            pd.DataFrame(validation_results).to_csv(partial_path, index=False)
        continue

    validation_results.append(result)
    case = spec['case']
    current_best = best_models.get(case)
    if current_best is None or result['val_macro_f1'] > current_best['result']['val_macro_f1']:
        best_models[case] = {'spec': spec, 'result': result, 'model': model}
        best_val_predictions[case] = val_pred

    print(
        f"val_macro_f1={result['val_macro_f1']:.4f}, "
        f"val_accuracy={result['val_accuracy']:.4f}, "
        f"n_features={result['n_features']}, "
        f"fit_time_sec={result['fit_time_sec']:.1f}"
    )

    if SAVE_AFTER_EACH_CANDIDATE:
        pd.DataFrame(validation_results).to_csv(partial_path, index=False)

validation_df = pd.DataFrame(validation_results)
if 'val_macro_f1' in validation_df.columns:
    validation_df = validation_df.sort_values('val_macro_f1', ascending=False, na_position='last').reset_index(drop=True)

validation_path = RESULTS_DIR / 'phase3_validation_results.csv'
validation_df.to_csv(validation_path, index=False)
print(f'Saved validation results to {validation_path}')
display(validation_df.head(20))

Phase 3 candidates:   0%|                        | 0/120 [00:00<?, ?it/s]


Running raw_pixels__baseline_0000: Raw Pixels (Baseline)



train/stretch/64/none: 100%|██████| 6605/6605 [00:00<00:00, 22370.19it/s]


('train', 64, 'stretch', 'none'): images=(6605, 64, 64, 3), labels=(6605,)



Raw gray: 100%|███████████████████| 6605/6605 [00:00<00:00, 83735.47it/s]


...

('val', 32, 'pad_square', 'none'): images=(824, 32, 32, 3), labels=(824,)



Raw gray: 100%|████████████████████| 824/824 [00:00<00:00, 125626.35it/s]


val/Raw Pixels (Baseline): X=(824, 1024), feature_time_sec=0.013


Phase 3 candidates:   4%|▌             | 5/120 [04:44<1:54:12, 59.58s/it]

val_macro_f1=0.9102, val_accuracy=0.9260, n_features=1024, fit_time_sec=17.3

Running raw_pixels__baseline_0005: Raw Pixels (Baseline)


In [ ]:
successful_df = validation_df.dropna(subset=['val_macro_f1']).copy()
best_per_group_df = (
    successful_df
    .sort_values('val_macro_f1', ascending=False)
    .groupby('case', as_index=False)
    .head(1)
    .sort_values('val_macro_f1', ascending=False)
    .reset_index(drop=True)
)

best_per_group_path = RESULTS_DIR / 'phase3_best_per_group_validation.csv'
best_per_group_df.to_csv(best_per_group_path, index=False)
print(f'Saved best validation candidate per group to {best_per_group_path}')
display(best_per_group_df)

if PHASE1_VALIDATION_PATH.exists():
    phase1_df = pd.read_csv(PHASE1_VALIDATION_PATH)
    phase1_selected = phase1_df[phase1_df['case'].isin(SELECTED_CASES)].copy()
    phase1_selected.insert(0, 'phase', 'phase_1')
    tuned_selected = best_per_group_df.copy()
    tuned_selected.insert(0, 'phase', 'phase_3_tuned')

    comparison_df = pd.concat(
        [phase1_selected, tuned_selected],
        ignore_index=True,
        sort=False,
    ).sort_values('val_macro_f1', ascending=False, na_position='last').reset_index(drop=True)
    comparison_path = RESULTS_DIR / 'phase3_vs_phase1_selected_features.csv'
    comparison_df.to_csv(comparison_path, index=False)
    print(f'Saved phase 3 vs phase 1 comparison to {comparison_path}')
    display(comparison_df[['phase', 'case', 'n_features', 'train_time_sec', 'val_accuracy', 'val_macro_f1', 'val_weighted_f1']])
else:
    print(f'Phase 1 validation CSV not found at {PHASE1_VALIDATION_PATH}; skipped comparison.')

In [ ]:
def spec_by_candidate_id(candidate_id: str) -> dict:
    for spec in search_plan:
        if spec['candidate_id'] == candidate_id:
            return spec
    raise KeyError(candidate_id)


def fit_spec_and_evaluate_test(spec: dict, prefix: str):
    X_train, y_train, blocks, block_weights = get_candidate_features(spec, 'train')
    X_test, y_test, _, _ = get_candidate_features(spec, 'test')

    model = build_pipeline(spec, blocks=blocks, block_weights=block_weights)
    fit_started = perf_counter()
    model.fit(X_train, y_train)
    fit_time = perf_counter() - fit_started

    predict_started = perf_counter()
    test_pred = model.predict(X_test)
    predict_time = perf_counter() - predict_started

    metrics = compute_metrics(y_test, test_pred)
    outputs = save_classification_outputs(prefix, y_test, test_pred)
    result = {
        **flatten_spec(spec),
        'n_features': X_train.shape[1],
        'train_samples': int(len(y_train)),
        'test_samples': int(len(y_test)),
        'fit_time_sec': fit_time,
        'test_predict_sec': predict_time,
        'test_inference_time_ms_per_image': (predict_time / len(y_test)) * 1000.0,
        **{f'test_{key}': value for key, value in metrics.items()},
        **{key: str(value) for key, value in outputs.items()},
    }
    return result, model


if successful_df.empty:
    raise ValueError('No successful validation candidates. Check errors in validation_df.')

best_overall_row = successful_df.sort_values('val_macro_f1', ascending=False).iloc[0]
best_overall_spec = spec_by_candidate_id(best_overall_row['candidate_id'])

print('Best overall validation candidate')
display(best_overall_row.to_frame(name='value'))

best_test_result, best_test_model = fit_spec_and_evaluate_test(
    best_overall_spec,
    prefix=f'test_best_overall_{slugify(best_overall_spec["case"])}',
)

test_results = [best_test_result]

if EVALUATE_TEST_FOR_EACH_GROUP:
    for _, row in best_per_group_df.iterrows():
        if row['candidate_id'] == best_overall_row['candidate_id']:
            continue
        spec = spec_by_candidate_id(row['candidate_id'])
        result, _ = fit_spec_and_evaluate_test(
            spec,
            prefix=f'test_best_group_{slugify(spec["case"])}',
        )
        test_results.append(result)

test_results_df = pd.DataFrame(test_results).sort_values('test_macro_f1', ascending=False).reset_index(drop=True)
test_results_path = RESULTS_DIR / 'phase3_test_results.csv'
test_results_df.to_csv(test_results_path, index=False)
print(f'Saved test results to {test_results_path}')
display(test_results_df)

best_model_path = RESULTS_DIR / 'best_phase3_svm_pipeline.joblib'
metadata_path = RESULTS_DIR / 'best_phase3_metadata.joblib'
joblib.dump(best_test_model, best_model_path)
joblib.dump(
    {
        'phase': 'phase_3',
        'best_candidate_id': best_overall_spec['candidate_id'],
        'best_case': best_overall_spec['case'],
        'best_spec': best_overall_spec,
        'class_names': class_names,
        'selected_cases': SELECTED_CASES,
        'search_mode': SEARCH_MODE,
        'max_candidates_per_group': MAX_CANDIDATES_PER_GROUP,
        'imblearn_available': IMBLEARN_AVAILABLE,
        'phase1_metadata_path': str(PHASE1_METADATA_PATH),
        'phase1_validation_path': str(PHASE1_VALIDATION_PATH),
    },
    metadata_path,
)
print(f'Saved best model to {best_model_path}')
print(f'Saved metadata to {metadata_path}')